# 02 — Preprocessing
**Steps 12–13:** Load exported GeoTIFFs, align datacube & label raster to same CRS/resolution/extent, and remove invalid background pixels.

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling as RS
import matplotlib.pyplot as plt

RAW_DIR       = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

YEARS = list(range(2018, 2026))

In [ ]:
# ── STEP 12: Align Datacube and Label Raster ─────────────────

def align_rasters(datacube_path, label_path, out_cube_path, out_label_path):
    """
    Reproject label raster to match datacube CRS, resolution, and extent.
    Saves aligned versions to processed directory.
    """
    with rasterio.open(datacube_path) as src_cube:
        cube_meta = src_cube.meta.copy()
        cube_data = src_cube.read()   # shape: (8, H, W)
        target_crs       = src_cube.crs
        target_transform = src_cube.transform
        target_height    = src_cube.height
        target_width     = src_cube.width

    with rasterio.open(label_path) as src_lbl:
        label_data = np.zeros((1, target_height, target_width), dtype=np.uint8)
        reproject(
            source        = rasterio.band(src_lbl, 1),
            destination   = label_data[0],
            dst_transform = target_transform,
            dst_crs       = target_crs,
            resampling    = Resampling.nearest
        )

    # Save aligned datacube
    with rasterio.open(out_cube_path, 'w', **cube_meta) as dst:
        dst.write(cube_data)

    # Save aligned labels
    lbl_meta = cube_meta.copy()
    lbl_meta.update({'count': 1, 'dtype': 'uint8'})
    with rasterio.open(out_label_path, 'w', **lbl_meta) as dst:
        dst.write(label_data)

    print(f'  ✅ Aligned: {os.path.basename(out_cube_path)} | {os.path.basename(out_label_path)}')
    return cube_data, label_data


# Align 2025 (primary year for training)
cube_path   = os.path.join(RAW_DIR, 'datacube_2025.tif')
label_path  = os.path.join(RAW_DIR, 'labels_2025.tif')
out_cube    = os.path.join(PROCESSED_DIR, 'datacube_2025_aligned.tif')
out_label   = os.path.join(PROCESSED_DIR, 'labels_2025_aligned.tif')

# cube_2025, labels_2025 = align_rasters(cube_path, label_path, out_cube, out_label)
print('Align function defined. Uncomment above line after GEE exports are downloaded.')

In [ ]:
# ── STEP 13: Remove Invalid Background Pixels ────────────────

def extract_valid_pixels(cube_array, label_array):
    """
    Mask out zero-padded background pixels outside Obuasi boundary.
    Returns flattened feature matrix X and label vector y for valid pixels only.
    Also returns the valid pixel mask for later reconstruction.

    Args:
        cube_array  : np.ndarray shape (8, H, W)
        label_array : np.ndarray shape (1, H, W) or (H, W)

    Returns:
        X     : np.ndarray shape (N_valid, 8)
        y     : np.ndarray shape (N_valid,)
        mask  : np.ndarray bool shape (H, W)  — True where valid
        shape : (H, W) original spatial shape
    """
    if label_array.ndim == 3:
        label_array = label_array[0]

    H, W = label_array.shape

    # Valid = label not 0 AND all spectral bands not all-zero
    band_sum = cube_array.sum(axis=0)   # (H, W)
    mask     = (label_array > 0) & (band_sum != 0)

    X = cube_array[:, mask].T    # (N_valid, 8)
    y = label_array[mask] - 1   # 0-indexed classes: 0..4

    print(f'  Valid pixels : {X.shape[0]:,} / {H * W:,} total')
    print(f'  Feature shape: {X.shape}')
    print(f'  Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

    return X, y, mask, (H, W)


# ── Quick Visualisation ───────────────────────────────────────
def preview_composite(cube_array, title='RGB Preview'):
    """Display a false-colour RGB composite (B4=Red, B3=Green, B2=Blue)."""
    rgb = cube_array[[2, 1, 0], :, :]   # B4, B3, B2
    rgb = np.clip(rgb / 0.3, 0, 1)      # stretch
    rgb = np.moveaxis(rgb, 0, -1)        # (H, W, 3)
    plt.figure(figsize=(8, 8))
    plt.imshow(rgb)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'outputs', 'rgb_preview_2025.png'), dpi=150)
    plt.show()

print('✅ Preprocessing functions defined.')

In [ ]:
# ── Save processed arrays ─────────────────────────────────────
# After running align_rasters and extract_valid_pixels, save:
#
# np.save(os.path.join(PROCESSED_DIR, 'X_valid.npy'), X)
# np.save(os.path.join(PROCESSED_DIR, 'y_valid.npy'), y)
# np.save(os.path.join(PROCESSED_DIR, 'valid_mask.npy'), mask)
# np.save(os.path.join(PROCESSED_DIR, 'spatial_shape.npy'), np.array(shape))
#
# These files are consumed by 03_graph_construction.
print('Save block ready. Run after GEE data is downloaded.')